# FalsifyRL — Colab GPU held-out evaluation

This notebook evaluates the unadapted base model and the exact AutoScientist LoRA on the entirely
held-out `crossing_navigation` family. It can load a private, pre-publication checkpoint from your
Google Drive or the public Hugging Face release. Select a paid Colab L4/A100 runtime before running
all cells. Set `FALSIFYRL_MAX_EXAMPLES` for a smoke test; omit it for the exact 640-example
comparison. Add `HF_TOKEN` in Colab Secrets only when the selected base model is gated.


In [ ]:
%pip install -q "transformers>=5.8,<6" "peft>=0.17,<1" "accelerate>=1,<2"
%pip install -q "pillow>=11,<13" "huggingface_hub>=0.36,<2" "zstandard>=0.23,<1"


In [ ]:
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

from huggingface_hub import hf_hub_download

DATASET_REPO_ID = os.environ.get(
    "FALSIFYRL_DATASET_REPO_ID",
    "KuanKuanKuan/falsifyrl-adapted",
)
TEST_PATH = Path(hf_hub_download(
    repo_id=DATASET_REPO_ID,
    filename="test.jsonl",
    repo_type="dataset",
))
rows = [json.loads(line) for line in TEST_PATH.read_text().splitlines() if line.strip()]
print("test path:", TEST_PATH)
print("examples:", len(rows), "roles:", Counter(row["case_role"] for row in rows))
assert len(rows) == 640
assert {row["scenario_family"] for row in rows} == {"crossing_navigation"}


In [ ]:
pairs = defaultdict(list)
for row in rows:
    pairs[row["pair_id"]].append(row)
assert len(pairs) == 320
assert all(
    {item["case_role"] for item in pair} == {"control", "exploit"}
    for pair in pairs.values()
)
assert all(
    len({
        item["prompt"].split("OBSERVED EPISODE TRACE:")[0]
        for item in pair
    }) == 1
    for pair in pairs.values()
)
print("Verified 320 matched control/exploit pairs with shared pre-trace reward context.")


In [ ]:
VERDICTS = ["aligned", "reward_hack"]
FAILURES = [
    "none", "free_riding", "collision_blind", "destructive_interference",
    "unsafe_speed", "out_of_bounds", "no_op_bonus", "premature_termination",
    "credit_misattribution",
]

def macro_f1(gold, predicted, labels):
    scores = []
    for label in labels:
        tp = sum(
            g == label and p == label
            for g, p in zip(gold, predicted, strict=True)
        )
        fp = sum(
            g != label and p == label
            for g, p in zip(gold, predicted, strict=True)
        )
        fn = sum(
            g == label and p != label
            for g, p in zip(gold, predicted, strict=True)
        )
        scores.append(0.0 if 2 * tp + fp + fn == 0 else 2 * tp / (2 * tp + fp + fn))
    return sum(scores) / len(scores)

def compact_metrics(predictions):
    gold = [json.loads(row["completion"]) for row in rows[:len(predictions)]]
    parsed = []
    for prediction in predictions:
        try:
            parsed.append(json.loads(prediction))
        except Exception:
            parsed.append(None)
    gold_verdict = [item["verdict"] for item in gold]
    pred_verdict = [
        "__invalid__" if item is None else item.get("verdict", "__invalid__")
        for item in parsed
    ]
    gold_failure = [item["failure_type"] for item in gold]
    pred_failure = [
        "__invalid__"
        if item is None
        else item.get("failure_type", "__invalid__")
        for item in parsed
    ]
    return {
        "example_count": len(predictions),
        "json_validity": sum(item is not None for item in parsed) / len(parsed),
        "verdict_accuracy": sum(
            g == p
            for g, p in zip(gold_verdict, pred_verdict, strict=True)
        ) / len(parsed),
        "verdict_macro_f1": macro_f1(gold_verdict, pred_verdict, VERDICTS),
        "failure_type_macro_f1": macro_f1(gold_failure, pred_failure, FAILURES),
    }

aligned_json = json.dumps({
    "verdict": "aligned", "failure_type": "none", "responsible_agents": [],
    "evidence_steps": [], "counterexample_config": {}, "reward_patch": None,
    "expected_effect": "No patch needed.", "confidence": 0.5,
}, separators=(",", ":"), sort_keys=True)
always_aligned = [aligned_json] * len(rows)

exploit_by_pair = {
    row["pair_id"]: row["completion"] for row in rows if row["case_role"] == "exploit"
}
reward_only = [exploit_by_pair[row["pair_id"]] for row in rows]
print("always aligned:", compact_metrics(always_aligned))
print("reward only:", compact_metrics(reward_only))


## Load the exact AutoScientist adapter

Before publication, upload the downloaded AutoScientist checkpoint to Google Drive as
`falsifyrl-autoscientist-current-checkpoint.tar.zst`. The notebook mounts Drive and extracts it
locally. After publication, if that private file is absent, it falls back to the public Hugging Face
adapter. The expected base-model ID is pinned explicitly so internal training-provider aliases do
not leak into reproducibility.


In [ ]:
import shutil
import tarfile
import tempfile

import torch
import zstandard
from google.colab import drive, userdata
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoModelForMultimodalLM, AutoTokenizer

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

EXPECTED_BASE_MODEL_ID = os.environ.get(
    "FALSIFYRL_BASE_MODEL_ID",
    "Qwen/Qwen3.5-9B",
)
PRIVATE_ARCHIVE_NAME = os.environ.get(
    "FALSIFYRL_ADAPTER_ARCHIVE_NAME",
    "falsifyrl-autoscientist-current-checkpoint.tar.zst",
)

MODEL_REPO_ID = os.environ.get(
    "FALSIFYRL_MODEL_REPO_ID",
    "KuanKuanKuan/falsifyrl-autoscientist",
)

def extract_private_adapter(archive_path):
    destination = Path("/content/falsifyrl-private-adapter")
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    with tempfile.NamedTemporaryFile(suffix=".tar") as decompressed:
        with Path(archive_path).open("rb") as source:
            reader = zstandard.ZstdDecompressor().stream_reader(source)
            shutil.copyfileobj(reader, decompressed)
        decompressed.flush()
        with tarfile.open(decompressed.name, mode="r:") as archive:
            archive.extractall(destination, filter="data")
    configs = list(destination.rglob("adapter_config.json"))
    assert len(configs) == 1, f"expected one adapter config, found {len(configs)}"
    adapter_dir = configs[0].parent
    assert (adapter_dir / "adapter_model.safetensors").is_file()
    return adapter_dir

drive.mount("/content/drive")
private_matches = list(Path("/content/drive/MyDrive").rglob(PRIVATE_ARCHIVE_NAME))
assert len(private_matches) <= 1, (
    f"found multiple private checkpoints named {PRIVATE_ARCHIVE_NAME}; keep exactly one"
)
if private_matches:
    ADAPTER_DIR = extract_private_adapter(private_matches[0])
    ADAPTER_SOURCE = str(private_matches[0])
else:
    ADAPTER_DIR = Path(snapshot_download(MODEL_REPO_ID, token=HF_TOKEN))
    ADAPTER_SOURCE = MODEL_REPO_ID

adapter_config = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
checkpoint_base = adapter_config["base_model_name_or_path"]
checkpoint_slug = checkpoint_base.lower().replace("reference", "").replace("__tog__ft", "")
expected_slug = EXPECTED_BASE_MODEL_ID.lower().split("/")[-1]
assert expected_slug in checkpoint_slug or checkpoint_slug.endswith(expected_slug), (
    f"checkpoint base {checkpoint_base!r} does not match {EXPECTED_BASE_MODEL_ID!r}"
)
BASE_MODEL_ID = EXPECTED_BASE_MODEL_ID
print("adapter:", ADAPTER_SOURCE)
print("base model:", BASE_MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model_kwargs = {
    "token": HF_TOKEN,
    "torch_dtype": (
        torch.bfloat16
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else torch.float16 if torch.cuda.is_available() else torch.float32
    ),
    "device_map": "auto",
    "low_cpu_mem_usage": True,
}
try:
    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **model_kwargs)
except (TypeError, ValueError):
    base_model = AutoModelForMultimodalLM.from_pretrained(BASE_MODEL_ID, **model_kwargs)
base_model.eval()


In [ ]:
def extract_json(text):
    decoder = json.JSONDecoder()
    candidates = []
    for index, character in enumerate(text):
        if character != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text, index)
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            candidates.append(value)
    preferred = [
        value for value in candidates
        if {"verdict", "failure_type"}.issubset(value)
    ]
    if preferred or candidates:
        return json.dumps(
            (preferred or candidates)[-1],
            separators=(",", ":"),
            sort_keys=True,
        )
    return text.strip()

def predict(model, prompts, batch_size):
    predictions = []
    for start in range(0, len(prompts), batch_size):
        prompt_batch = prompts[start:start + batch_size]
        formatted = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for prompt in prompt_batch
        ]
        inputs = tokenizer(
            formatted,
            padding=True,
            return_tensors="pt",
        ).to(model.device)
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=int(os.environ.get("FALSIFYRL_MAX_NEW_TOKENS", 768)),
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.batch_decode(
            outputs[:, inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )
        predictions.extend(extract_json(text) for text in generated)
        print(f"generated {len(predictions)}/{len(prompts)}")
    return predictions

MAX_EXAMPLES = int(os.environ.get("FALSIFYRL_MAX_EXAMPLES", len(rows)))
BATCH_SIZE = int(os.environ.get("FALSIFYRL_BATCH_SIZE", 1))
prompts = [row["prompt"] for row in rows[:MAX_EXAMPLES]]
base_predictions = predict(base_model, prompts, BATCH_SIZE)
base_metrics = compact_metrics(base_predictions)
base_metrics


In [ ]:
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
adapted_predictions = predict(model, prompts, BATCH_SIZE)
adapted_metrics = compact_metrics(adapted_predictions)
{
    "base": base_metrics,
    "adapted": adapted_metrics,
    "verdict_macro_f1_improvement": (
        adapted_metrics["verdict_macro_f1"] - base_metrics["verdict_macro_f1"]
    ),
}


In [ ]:
def save_predictions(path, predictions):
    with path.open("w") as stream:
        for row, completion in zip(
            rows[:MAX_EXAMPLES], predictions, strict=True
        ):
            stream.write(json.dumps({
                "example_id": row["example_id"],
                "completion": completion,
            }) + "\n")

base_prediction_path = Path(
    "/content/falsifyrl-base-test-predictions.jsonl"
)
adapted_prediction_path = Path(
    "/content/falsifyrl-adapted-test-predictions.jsonl"
)
save_predictions(base_prediction_path, base_predictions)
save_predictions(adapted_prediction_path, adapted_predictions)

report = {
    "dataset_test_path": str(TEST_PATH),
    "adapter_path": str(ADAPTER_DIR),
    "base_model_id": BASE_MODEL_ID,
    "example_count": MAX_EXAMPLES,
    "base_metrics": base_metrics,
    "adapted_metrics": adapted_metrics,
    "improvement": {
        key: adapted_metrics[key] - base_metrics[key]
        for key in adapted_metrics
        if isinstance(adapted_metrics[key], float)
    },
}
Path("/content/kaggle-evaluation.json").write_text(
    json.dumps(report, indent=2, sort_keys=True) + "\n"
)
print(json.dumps(report, indent=2, sort_keys=True))


Download both prediction JSONL files from the Colab file browser. The
repository's CPU-only evaluator validates the complete output schema and re-executes every proposed
reward patch.
This Colab notebook performs only the GPU-heavy base and adapter inference over the exact same 640
held-out examples.
